# AZR Model Trainer v2 — Google Colab

**Neural network trainer with web UI, GPU acceleration, dataset catalog (110 datasets)**

---

### How to use:
1. **Runtime → Change runtime type → GPU (T4)**
2. Run all cells in order (Ctrl+F9)
3. Click the **cloudflare link** at the end to open the web interface

**GPU works automatically on Colab!** No "Activate GPU" needed — Tesla T4 + CUDA is detected instantly.

---

## 1. Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f'VRAM: {vram / 1024**3:.1f} GB')
else:
    print('WARNING: GPU not found! Go to Runtime -> Change runtime type -> GPU')

## 2. Clone repository & install dependencies

In [ ]:
import os

# Clone repo
if not os.path.exists('/content/ai-neural-network-project'):
    !git clone https://github.com/Slavikpro557/ai-neural-network-project.git
else:
    print('Repo already cloned, pulling latest...')
    !cd /content/ai-neural-network-project && git pull

# Install only missing packages (torch, numpy, psutil already on Colab)
!pip install -q fastapi uvicorn python-multipart pydantic PyMuPDF 2>&1 | tail -5

# Install cloudflared (tunnel, no registration needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('\nDone! cloudflared + dependencies installed.')

## 3. Start AZR Trainer server + tunnel

This cell starts the server and creates a **Cloudflare tunnel** (no registration, no tokens needed).

Click the link that appears below to open the web interface.

In [ ]:
import subprocess
import time
import re
import threading
import torch

os.chdir('/content/ai-neural-network-project')

# Kill any existing server
!kill $(lsof -t -i:8000) 2>/dev/null; true
time.sleep(1)

# Start server in background
server_process = subprocess.Popen(
    ['python', 'server_with_datasets.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    cwd='/content/ai-neural-network-project'
)

print('Starting server...')
time.sleep(5)

# Check if server started
if server_process.poll() is None:
    print('Server is running!')
else:
    print('ERROR: Server failed to start. Check logs below:')
    print(server_process.stdout.read().decode('utf-8', errors='replace')[:3000])
    raise RuntimeError('Server failed to start')

# Start cloudflare tunnel
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait for tunnel URL
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
url_found = False

# Read stderr in a thread to avoid blocking
tunnel_url = [None]

def read_tunnel():
    for line in tunnel_process.stderr:
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url[0] = match.group(0)
            return

t = threading.Thread(target=read_tunnel, daemon=True)
t.start()
t.join(timeout=15)

if tunnel_url[0]:
    print()
    print('=' * 60)
    print(f'  AZR Model Trainer is running!')
    print(f'  GPU: {gpu_name}')
    print(f'')
    print(f'  Open this link:')
    print(f'  {tunnel_url[0]}')
    print(f'')
    print(f'  (keep this cell running!)')
    print('=' * 60)
else:
    print('WARNING: Could not get tunnel URL. Try running this cell again.')
    print('Server is still running on localhost:8000')

## 4. Server logs (optional)
Run this cell to see server output if something goes wrong:

In [ ]:
# Show server status
print('Server PID:', server_process.pid)
print('Status:', 'Running' if server_process.poll() is None else 'Stopped')
print('Tunnel:', 'Running' if tunnel_process.poll() is None else 'Stopped')
if tunnel_url[0]:
    print(f'URL: {tunnel_url[0]}')

## 5. Stop server (when done)

In [ ]:
# STOP SERVER — run this cell MANUALLY when you're done
# (safe to skip during "Run all")
STOP_SERVER = False  # Change to True and run this cell to stop

if STOP_SERVER:
    tunnel_process.terminate()
    server_process.terminate()
    print('Server stopped. Tunnel closed.')
else:
    print('Server is still running. Set STOP_SERVER = True and re-run to stop.')